## EJERCICIO 1 - Sin Binario

In [1]:
import numpy as np
import random
import copy

class Individuo:
    def __init__(self, limites):
        self.genotipo = []
        for lim_inf, lim_sup in limites:
            valor = random.uniform(lim_inf, lim_sup)
            self.genotipo.append(valor)
        self.fitness = None

    def mutar(self, tasa_mutacion, limites):
        for i in range(len(self.genotipo)):
            if random.random() < tasa_mutacion:               
                self.genotipo[i] += np.random.normal(0, 0.1 * (limites[i][1] - limites[i][0]))
                self.genotipo[i] = np.clip(self.genotipo[i], limites[i][0], limites[i][1])

    def cruzar(self, madre):
        hijo1 = copy.deepcopy(self)
        hijo2 = copy.deepcopy(madre)
        # modificacion optativa (50%)
        for i in range(len(self.genotipo)):
            if random.random() < 0.5:
                promedio = (self.genotipo[i] + madre.genotipo[i]) / 2
                hijo1.genotipo[i] = promedio
                hijo2.genotipo[i] = self.genotipo[i]
        return hijo1, hijo2

class AlgoritmoGenetico:
    def __init__(self, funcion_objetivo, limites, tam_poblacion=50, generaciones=100, tasa_mutacion=0.1):
        self.funcion_objetivo = funcion_objetivo
        self.limites = limites
        self.tam_poblacion = tam_poblacion
        self.generaciones = generaciones
        self.tasa_mutacion = tasa_mutacion
        self.poblacion = []
        self.mejor_individuo = None

    def inicializar_poblacion(self):
        self.poblacion = [Individuo(self.limites) for _ in range(self.tam_poblacion)]
        for ind in self.poblacion:
            self.evaluar(ind)

    # metodo por competencias
    def seleccion_comp(self, k=3):
        competencia = random.sample(self.poblacion, k)
        def obtener_fitness(ind):
            return ind.fitness
        return min(competencia, key=obtener_fitness)

    def evaluar(self, individuo):
        individuo.fitness = self.funcion_objetivo(individuo.genotipo)
        if self.mejor_individuo is None or individuo.fitness < self.mejor_individuo.fitness:
            self.mejor_individuo = copy.deepcopy(individuo)
    
    def evolucionar(self):
        nueva_poblacion = []
        
        # Elitismo
        nueva_poblacion.append(copy.deepcopy(self.mejor_individuo))

        while len(nueva_poblacion) < self.tam_poblacion:
            padre = self.seleccion_comp()
            madre = self.seleccion_comp()
            
            hijo1, hijo2 = padre.cruzar(madre)
            
            hijo1.mutar(self.tasa_mutacion, self.limites)
            hijo2.mutar(self.tasa_mutacion, self.limites)
            
            self.evaluar(hijo1)
            self.evaluar(hijo2)

            nueva_poblacion.append(hijo1)
            nueva_poblacion.append(hijo2)
            
        self.poblacion = nueva_poblacion[:self.tam_poblacion]

    def ejecutar(self):
        self.inicializar_poblacion()
        for _ in range(self.generaciones):
            self.evolucionar()
        return self.mejor_individuo



# GRADIENTE DESCENDENTE
def gradiente_descendente(funcion, x0, limites, cte=0.01, iteraciones=1000):
    x = np.array(x0)
    h = 1e-5
    
    for _ in range(iteraciones):
        grad = np.zeros_like(x)

        # Diferencias finitas
        for i in range(len(x)):
            xh = x.copy()
            xh[i] += h
            grad[i] = (funcion(xh) - funcion(x)) / h
            
        x = x - cte * grad
            
    return x, funcion(x)


# FUNCIONES 1 Y 2
def f1(x):
    val = x[0]
    return -val * np.sin(np.sqrt(np.abs(val)))

def f2(x):
    x_val, y_val = x[0], x[1]
    term1 = (x_val**2 + y_val**2)**0.25
    term2 = np.sin(50 * (x_val**2 + y_val**2)**0.1)**2 + 1
    return term1 * term2



if __name__ == "__main__":
    print("-- Resultados Funcion 1: ")
    ag1 = AlgoritmoGenetico(f1, limites=[(-512, 512)], tam_poblacion=100, generaciones=150)
    mejor_ag1 = ag1.ejecutar()
    
    x0_gd = [0.0] 
    x_gd1, f_gd1 = gradiente_descendente(f1, x0_gd, limites=[(-512, 512)])

    print(f"Gentico --> Mínimo en x={mejor_ag1.genotipo[0]:.4f} | f(x)={mejor_ag1.fitness:.4f}")
    print(f"Gradiente --> Mínimo en x={x_gd1[0]:.4f} | f(x)={f_gd1:.4f}\n")



    print("-- Resultados Funcion 2: ")
    ag2 = AlgoritmoGenetico(f2, limites=[(-100, 100), (-100, 100)], tam_poblacion=150, generaciones=200)
    mejor_ag2 = ag2.ejecutar()

    x0_gd2 = [50.0, 50.0]
    x_gd2, f_gd2 = gradiente_descendente(f2, x0_gd2, limites=[(-100, 100), (-100, 100)])

    print(f"Gentico --> Mínimo en x={mejor_ag2.genotipo[0]:.4f}, y={mejor_ag2.genotipo[1]:.4f} | f(x,y)={mejor_ag2.fitness:.4f}")
    print(f"Gradiente --> Mínimo en x={x_gd2[0]:.4f}, y={x_gd2[1]:.4f} | f(x,y)={f_gd2:.4f}")

-- Resultados Funcion 1: 
Gentico --> Mínimo en x=420.9687 | f(x)=-418.9829
Gradiente --> Mínimo en x=5.0801 | f(x)=-3.9402

-- Resultados Funcion 2: 
Gentico --> Mínimo en x=0.0000, y=-0.0000 | f(x,y)=0.0035
Gradiente --> Mínimo en x=47.9946, y=47.9946 | f(x,y)=8.2396
